In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold
 
# ─────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")
 
print("Train shape:", train.shape)
print("Test shape: ", test.shape)
 
HORIZONS = [12, 24, 48, 72]

Train shape: (221, 37)
Test shape:  (95, 35)


In [2]:
# ─────────────────────────────────────────────────────────────────
# 2. SURVIVAL HELPERS  (FOR BRIER CALCULATION)
# ─────────────────────────────────────────────────────────────────
def y_horizon(df, H):
    if H == 72:
        return df["event"].astype(int)
    return ((df["event"] == 1) & (df["time_to_hit_hours"] <= H)).astype(int)
 
def known_mask(df, H):
    if H == 72:
        return np.ones(len(df), dtype=bool)
    return ~((df["event"] == 0) & (df["time_to_hit_hours"] < H))
 
def weighted_brier(df, probs_df,
                   horizons=(24, 48, 72),
                   weights={24: 0.3, 48: 0.4, 72: 0.3}):
    briers = {}
    for H in horizons:
        col = f"prob_{H}h"
        m   = known_mask(df, H)
        briers[H] = brier_score_loss(
            y_horizon(df, H)[m], probs_df.loc[m, col])
    return sum(weights[H] * briers[H] for H in horizons), briers
 
def enforce_monotonic(df_probs):
    out = df_probs.copy()
    out["prob_24h"] = np.maximum(out["prob_24h"], out["prob_12h"])
    out["prob_48h"] = np.maximum(out["prob_48h"], out["prob_24h"])
    out["prob_72h"] = np.maximum(out["prob_72h"], out["prob_48h"])
    return out.clip(0, 1)
 

In [3]:
# ─────────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
#
# Strategy:
#   - Keep ALL 34 original raw columns — no pruning
#   - Add domain-driven features from what the data showed:
#       * dist_min splits perfectly at 8.7km (0% hit rate beyond)
#       * 196/221 fires have zero measured growth (26 unique values)
#       * low_temporal_resolution = 3x hit rate multiplier
#       * alignment_abs has real signal only for near fires
#       * spread_bearing_cos: hits=0.72 vs censored=0.97
# ─────────────────────────────────────────────────────────────────
def feature_engineering(df):
    df  = df.copy()
    eps = 1e-6
 
    dist    = df["dist_min_ci_0_5h"].clip(lower=1)
    dist_km = dist / 1000.0
    closing = df["closing_speed_m_per_h"]
    radial  = df["radial_growth_rate_m_per_h"]
    eff_spd = (closing + radial).clip(lower=0)
    hi_res  = (df["low_temporal_resolution_0_5h"] == 0).astype(float)
 
    # ── Original v2 engineered features ────────────────
    df["log_distance"]       = np.log1p(dist)
    df["dist_km"]            = dist_km
    df["inv_distance"]       = 1.0 / (dist_km + 0.1)
    df["eta_hours"]          = dist / np.maximum(closing, eps)
    df["eta_effective"]      = dist / np.maximum(eff_spd,  eps)
    fire_area_m2             = df["area_first_ha"] * 10000.0
    df["fire_radius_est"]    = np.sqrt(fire_area_m2 / np.pi)
    df["radius_to_distance"] = df["fire_radius_est"] / np.maximum(dist, eps)
    df["threat_score"]       = df["alignment_abs"] * closing / np.log1p(dist)
    df["zone_critical"]      = (dist < 5000).astype(int)
 
    # ── Distance zone flags ───────────────────────────────────
    # From data: fires >8.7km → 0% hit rate; 0-8.7km → 77% hit rate
    df["near_zone"]          = (dist_km <   8.7).astype(float)
    df["mid_zone"]           = ((dist_km >= 8.7) & (dist_km < 83.4)).astype(float)
    df["far_zone"]           = (dist_km >= 83.4).astype(float)
    df["log_dist_near"]      = np.log1p(dist_km) * df["near_zone"]
    df["inv_dist_near"]      = df["inv_distance"] * df["near_zone"]
 
    # ── Fire movement flags ───────────────────────────────────
    # 196/221 fires have zero growth — any movement is a strong signal
    # Zero-growth event rate: 26%;  non-zero growth event rate: 72%
    df["fire_moving"]        = (df["area_growth_abs_0_5h"] != 0).astype(float)
    df["has_centroid_move"]  = (df["centroid_displacement_m"] > 0).astype(float)
    df["any_movement"]       = ((df["fire_moving"] + df["has_centroid_move"]) > 0).astype(float)
    df["moving_x_near"]      = df["fire_moving"] * df["near_zone"]
 
    # ── High-res data interactions ───────────────────────────
    # low_temporal_resolution=0 (hi-res): 60% hit rate
    # low_temporal_resolution=1 (lo-res): 20% hit rate — 3x difference
    df["hi_res_x_near"]      = hi_res * df["near_zone"]
    df["hi_res_x_inv_dist"]  = hi_res * df["inv_distance"]
    df["hi_res_x_alignment"] = hi_res * df["alignment_abs"]
    df["hi_res_x_moving"]    = hi_res * df["fire_moving"]
 
    # ── Alignment only meaningful for near fires ──────────────
    # alignment_abs: hits mean=0.34 vs censored mean=0.10
    # but only matters when fire is close enough to threaten
    df["alignment_near"]     = df["alignment_abs"]   * df["near_zone"]
    df["alignment_x_dist"]   = df["alignment_abs"]   * df["inv_distance"]
 
    # ── Bearing threat ────────────────────────────────────────
    # spread_bearing_cos: hits mean=0.72 vs censored mean=0.97
    # Low cos = fire spreading toward zone; invert to make it a threat score
    df["bearing_threat"]     = 1.0 - df["spread_bearing_cos"]
    df["bearing_threat_near"]= df["bearing_threat"] * df["near_zone"]
 
    # ── ETA horizon features ──────────────────────────────────
    df["eta_clip"]           = df["eta_effective"].clip(upper=200)
    df["log_eta"]            = np.log1p(df["eta_clip"])
    df["dist_margin_12h"]    = (eff_spd * 12  - dist) / 1000.0
    df["dist_margin_24h"]    = (eff_spd * 24  - dist) / 1000.0
    df["on_track_12h"]       = (eff_spd * 12  > dist).astype(float)
    df["on_track_24h"]       = (eff_spd * 24  > dist).astype(float)
 
    # ── Growth rate features (meaningful only for movers) ─────
    df["log_growth_rate"]    = np.log1p(
        df["area_growth_rate_ha_per_h"].clip(lower=0))
    df["growth_x_near"]      = df["log_growth_rate"] * df["near_zone"]
    df["growth_x_inv_dist"]  = df["log_growth_rate"] * df["inv_distance"]
 
    # ── Temporal coverage quality ────────────────────────────
    df["perim_density"]      = (df["num_perimeters_0_5h"] /
                                (df["dt_first_last_0_5h"] + eps))
    df["log_perim_density"]  = np.log1p(df["perim_density"])
 
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df
 
 
train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)
 
# Use ALL columns — no need for any pruning becasue this dataset is small (regularization handles noise on small datasets)
FEATURE_COLS = [c for c in train_fe.columns
                if c not in ["event_id", "time_to_hit_hours", "event"]]
print(f"\nTotal features: {len(FEATURE_COLS)}")


Total features: 71


In [4]:
# ─────────────────────────────────────────────────────────────────
# 4. PARAM GRID SEARCH
#    Grid uses full pipeline (with isotonic) so comparison is fair
# ─────────────────────────────────────────────────────────────────
print("\n=== Param grid search ===")
 
param_grid = [
    dict(max_depth=3, num_leaves=11, n_estimators=400, learning_rate=0.04),  # v2 baseline
    dict(max_depth=4, num_leaves=15, n_estimators=500, learning_rate=0.03),
    dict(max_depth=5, num_leaves=20, n_estimators=400, learning_rate=0.04),
    dict(max_depth=3, num_leaves=15, n_estimators=600, learning_rate=0.03),
    dict(max_depth=4, num_leaves=11, n_estimators=500, learning_rate=0.03),
    dict(max_depth=6, num_leaves=20, n_estimators=300, learning_rate=0.05),
    dict(max_depth=3, num_leaves=11, n_estimators=600, learning_rate=0.03),
]
 
# Stronger regularization + lower colsample to stop dist_min from
# dominating every tree and hiding the zone/movement/bearing signals
BASE = dict(
    min_child_samples=25,
    reg_lambda=3.0,
    reg_alpha=0.5,
    subsample=0.85,
    colsample_bytree=0.7,
    objective="binary",
    n_jobs=-1,
    verbose=-1,
)
 
best_wb     = 999.0
best_params = param_grid[0]
_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
for p in param_grid:
    _raw = pd.DataFrame(0.0, index=train_fe.index,
                        columns=[f"prob_{H}h" for H in HORIZONS])
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        XH  = train_fe.loc[m, FEATURE_COLS]
        yH  = y_horizon(train_fe, H).loc[m]
        for _, (tr_i, va_i) in enumerate(_cv.split(XH, yH)):
            mdl = LGBMClassifier(**BASE, **p)
            mdl.fit(XH.iloc[tr_i], yH.iloc[tr_i],
                    eval_set=[(XH.iloc[va_i], yH.iloc[va_i])],
                    eval_metric="binary_logloss",
                    callbacks=[early_stopping(50, verbose=False),
                               log_evaluation(0)])
            _raw.loc[XH.index[va_i], col] = \
                mdl.predict_proba(XH.iloc[va_i])[:, 1]
 
    # Apply full isotonic pipeline — same as seed loop
    _cal = _raw.copy()
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(_raw.loc[m, col], y_horizon(train_fe, H).loc[m])
        _cal[col] = iso.transform(_raw[col])
    _cal = enforce_monotonic(_cal)
 
    wb, _ = weighted_brier(train_fe, _cal)
    marker = " ← best" if wb < best_wb else ""
    print(f"  depth={p['max_depth']} leaves={p['num_leaves']:2d} "
          f"lr={p['learning_rate']} est={p['n_estimators']} "
          f"→ WB={wb:.6f}{marker}")
    if wb < best_wb:
        best_wb, best_params = wb, p
 
print(f"\nBest params: {best_params}")
print(f"Grid search best WB: {best_wb:.6f}")
print(f"v2 baseline to beat: 0.009075")


=== Param grid search ===
  depth=3 leaves=11 lr=0.04 est=400 → WB=0.011000 ← best
  depth=4 leaves=15 lr=0.03 est=500 → WB=0.010885 ← best
  depth=5 leaves=20 lr=0.04 est=400 → WB=0.011003
  depth=3 leaves=15 lr=0.03 est=600 → WB=0.010926
  depth=4 leaves=11 lr=0.03 est=500 → WB=0.010885
  depth=6 leaves=20 lr=0.05 est=300 → WB=0.010936
  depth=3 leaves=11 lr=0.03 est=600 → WB=0.010926

Best params: {'max_depth': 4, 'num_leaves': 15, 'n_estimators': 500, 'learning_rate': 0.03}
Grid search best WB: 0.010885
v2 baseline to beat: 0.009075


In [5]:
# ─────────────────────────────────────────────────────────────────
# 5. SEED BAGGING  (30 seeds, per-horizon colsample)
# ─────────────────────────────────────────────────────────────────
SEEDS = list(range(42, 72))   # 30 seeds — free variance reduction
 
BEST_PARAMS = {
    "n_estimators":      best_params["n_estimators"],
    "learning_rate":     best_params["learning_rate"],
    "max_depth":         best_params["max_depth"],
    "num_leaves":        best_params["num_leaves"],
    "min_child_samples": 25,
    "reg_lambda":        3.0,
    "reg_alpha":         0.5,
    "subsample":         0.85,
}
 
# Lower colsample at short horizons forces the model to find
# non-dist_min signal where discrimination is harder
COLSAMPLE_BY_H = {12: 0.60, 24: 0.65, 48: 0.70, 72: 0.75}
 
test_sum  = pd.DataFrame(0.0, index=test_fe.index,
                         columns=[f"prob_{H}h" for H in HORIZONS])
oof_sum   = pd.DataFrame(0.0, index=train_fe.index,
                         columns=[f"prob_{H}h" for H in HORIZONS])
seed_scores = []
 
for seed in SEEDS:
    print(f"\n===== SEED {seed} =====")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
 
    oof_raw  = pd.DataFrame(0.0, index=train_fe.index,
                            columns=[f"prob_{H}h" for H in HORIZONS])
    test_raw = pd.DataFrame(0.0, index=test_fe.index,
                            columns=[f"prob_{H}h" for H in HORIZONS])
 
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        XH  = train_fe.loc[m, FEATURE_COLS]
        yH  = y_horizon(train_fe, H).loc[m]
 
        for fold, (tr_idx, va_idx) in enumerate(cv.split(XH, yH), 1):
            model = LGBMClassifier(
                objective="binary",
                colsample_bytree=COLSAMPLE_BY_H[H],
                random_state=seed + 1000 * H + fold,
                n_jobs=-1,
                verbose=-1,
                **BEST_PARAMS
            )
            model.fit(
                XH.iloc[tr_idx], yH.iloc[tr_idx],
                eval_set=[(XH.iloc[va_idx], yH.iloc[va_idx])],
                eval_metric="binary_logloss",
                callbacks=[early_stopping(50, verbose=False),
                           log_evaluation(0)]
            )
            oof_raw.loc[XH.index[va_idx], col] = \
                model.predict_proba(XH.iloc[va_idx])[:, 1]
            test_raw[col] += \
                model.predict_proba(test_fe[FEATURE_COLS])[:, 1] / cv.get_n_splits()
 
    # Isotonic calibration on ALL horizons (including 72h)
    oof_cal  = oof_raw.copy()
    test_cal = test_raw.copy()
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_raw.loc[m, col], y_horizon(train_fe, H).loc[m])
        oof_cal[col]  = iso.transform(oof_raw[col])
        test_cal[col] = iso.transform(test_raw[col])
 
    oof_cal  = enforce_monotonic(oof_cal)
    test_cal = enforce_monotonic(test_cal)
 
    wb, briers = weighted_brier(train_fe, oof_cal)
    seed_scores.append(wb)
    print(f"  OOF WB: {wb:.6f} | "
          f"24h={briers[24]:.4f} "
          f"48h={briers[48]:.4f} "
          f"72h={briers[72]:.6f}")
 
    test_sum += test_cal
    oof_sum  += oof_cal


===== SEED 42 =====
  OOF WB: 0.010859 | 24h=0.0168 48h=0.0146 72h=0.000000

===== SEED 43 =====
  OOF WB: 0.010039 | 24h=0.0206 48h=0.0094 72h=0.000402

===== SEED 44 =====
  OOF WB: 0.012510 | 24h=0.0230 48h=0.0140 72h=0.000000

===== SEED 45 =====
  OOF WB: 0.009657 | 24h=0.0219 48h=0.0077 72h=0.000000

===== SEED 46 =====
  OOF WB: 0.010685 | 24h=0.0206 48h=0.0112 72h=0.000000

===== SEED 47 =====
  OOF WB: 0.011704 | 24h=0.0214 48h=0.0132 72h=0.000000

===== SEED 48 =====
  OOF WB: 0.009608 | 24h=0.0207 48h=0.0085 72h=0.000000

===== SEED 49 =====
  OOF WB: 0.009250 | 24h=0.0180 48h=0.0096 72h=0.000000

===== SEED 50 =====
  OOF WB: 0.010592 | 24h=0.0229 48h=0.0091 72h=0.000283

===== SEED 51 =====
  OOF WB: 0.010135 | 24h=0.0201 48h=0.0103 72h=0.000000

===== SEED 52 =====
  OOF WB: 0.010833 | 24h=0.0198 48h=0.0122 72h=0.000000

===== SEED 53 =====
  OOF WB: 0.010141 | 24h=0.0195 48h=0.0107 72h=0.000000

===== SEED 54 =====
  OOF WB: 0.008062 | 24h=0.0175 48h=0.0070 72h=0.000000

In [6]:
# ─────────────────────────────────────────────────────────────────
# 6. FINAL PREDICTIONS
# ─────────────────────────────────────────────────────────────────
final_test = enforce_monotonic(test_sum / len(SEEDS))
final_oof  = enforce_monotonic(oof_sum  / len(SEEDS))
 
final_wb, final_briers = weighted_brier(train_fe, final_oof)
 
print(f"\n{'='*50}")
print(f"Final OOF WB:  {final_wb:.6f}")
print(f"  24h Brier:   {final_briers[24]:.6f}")
print(f"  48h Brier:   {final_briers[48]:.6f}")
print(f"  72h Brier:   {final_briers[72]:.6f}")
print(f"Seed avg WB:   {np.mean(seed_scores):.6f} ± {np.std(seed_scores):.6f}")
print(f"v2 baseline:   0.009075")
print(f"Delta:         {final_wb - 0.009075:+.6f} "
      f"({'BETTER' if final_wb < 0.009075 else 'WORSE'})")
print(f"{'='*50}")


Final OOF WB:  0.009752
  24h Brier:   0.019328
  48h Brier:   0.009851
  72h Brier:   0.000045
Seed avg WB:   0.010843 ± 0.001897
v2 baseline:   0.009075
Delta:         +0.000677 (WORSE)


In [7]:
print("\n=== Prediction health check ===")
for H in HORIZONS:
    col = f"prob_{H}h"
    s   = final_oof[col]
    flag = " *** COLLAPSED ***" if s.std() < 0.05 else ""
    print(f"  {col}: mean={s.mean():.4f} std={s.std():.4f} "
          f"min={s.min():.4f} max={s.max():.4f}{flag}")


=== Prediction health check ===
  prob_12h: mean=0.2217 std=0.3554 min=0.0000 max=0.9857
  prob_24h: mean=0.2903 std=0.4330 min=0.0000 max=0.9972
  prob_48h: mean=0.3058 std=0.4539 min=0.0000 max=1.0000
  prob_72h: mean=0.3134 std=0.4637 min=0.0000 max=1.0000


In [9]:
# ─────────────────────────────────────────────────────────────────
# 7. SUBMISSION
# ─────────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h":  final_test["prob_12h"].values,
    "prob_24h":  final_test["prob_24h"].values,
    "prob_48h":  final_test["prob_48h"].values,
    "prob_72h":  final_test["prob_72h"].values,
})
submission.to_csv("submission_v5.csv", index=False)
print("\nSaved: submission_v5.csv")
print(submission[["prob_12h", "prob_24h",
                  "prob_48h", "prob_72h"]].describe().round(4))


Saved: submission_v5.csv
       prob_12h  prob_24h  prob_48h  prob_72h
count   95.0000   95.0000   95.0000   95.0000
mean     0.1765    0.2787    0.2927    0.2952
std      0.3005    0.4329    0.4544    0.4580
min      0.0000    0.0000    0.0000    0.0000
25%      0.0000    0.0000    0.0000    0.0000
50%      0.0000    0.0000    0.0000    0.0000
75%      0.4174    0.9150    0.9952    1.0000
max      0.9641    0.9882    1.0000    1.0000
